# AI 폰트 파이프라인 v2 — 층화 측정 + 다중생성 + 이중 선택기

**Run all 한 번 (~15 분, T4)** 으로 폰트 트랙의 갈림길을 결정하는 데이터가 나옴.

v1 대비 재설계 3 가지:
1. **층화 글자**: 단순(가나사) + 받침(한글정) + 겹받침(값삶닭).  11,172 자의
   96.4% 가 받침 글자 — 쉬운 글자만으로는 폰트 가능성 판정 불가
2. **시드 고정**: sample.py 패치로 배치별 deterministic seed → 재현 가능한 측정
   (지금까지 모든 비교는 무작위 노이즈에 잠겨 있었음)
3. **이중 선택기**: 11,172 자 프로토타입 구조 분류기 (주) + easyocr (보조).
   easyocr 가 고립 글자에 실패해도 파이프라인이 죽지 않음

마지막 셀이 **KS X 1001 2,350 자 / 전체 11,172 자 생성 비용을 자동 외삽** —
이 숫자가 Phase 1 진행 여부를 결정.


## 1. 환경 → 생성까지

In [ ]:
!nvidia-smi | head -12
import os
if not os.path.exists('Korean-Diff-Font'):
    !git clone -q https://github.com/ORI-Muchim/Korean-Diff-Font.git
%cd Korean-Diff-Font

In [ ]:
!apt-get install -y libopenmpi-dev 2>&1 | tail -1
!pip install -q tqdm opencv-python scikit-learn pillow tensorboardX 'blobfile>=1.0.5' mpi4py attrdict pyyaml easyocr 2>&1 | tail -2

# attrdict Python 3.12 호환 패치
import glob, sys
for f in glob.glob('/usr/local/lib/python3.*/dist-packages/attrdict/*.py'):
    s = open(f).read()
    s2 = (s.replace('from collections import Mapping', 'from collections.abc import Mapping')
            .replace('from collections import MutableMapping', 'from collections.abc import MutableMapping')
            .replace('from collections import Sequence', 'from collections.abc import Sequence'))
    if s2 != s:
        open(f, 'w').write(s2)
for m in [k for k in list(sys.modules) if k == 'attrdict' or k.startswith('attrdict.')]:
    del sys.modules[m]
from attrdict import AttrDict
print('deps OK:', AttrDict({'k': 'v'}).k)

In [ ]:
!mkdir -p trained_models
!wget -q -O trained_models/ema_0.9999_800000.pt \
    https://github.com/ORI-Muchim/Korean-Diff-Font/releases/download/1.0/ema_0.9999_800000.pt
!ls -lh trained_models/

In [ ]:
import urllib.request
from PIL import Image, ImageDraw, ImageFont

TARGET_FONT_URL = 'https://github.com/google/fonts/raw/main/ofl/nanumgothic/NanumGothic-Regular.ttf'
REF_CHAR = '가'

# 층화 테스트 셋 — 난이도 3 클래스 x 3 글자
DIFF = {'단순(받침无)': '가나사', '일반(홑받침)': '한글정', '복합(겹받침)': '값삶닭'}
GEN_CHARS = ''.join(DIFF.values())   # 가나사한글정값삶닭

urllib.request.urlretrieve(TARGET_FONT_URL, 'target_font.ttf')

# 학습(font2img.py) 정합 렌더: 128 캔버스, 폰트크기 100, 고정 오프셋 14
def render_train(font_path, ch, canvas=128, size=100, off=14):
    f = ImageFont.truetype(font_path, size)
    img = Image.new('RGB', (canvas, canvas), (255, 255, 255))
    ImageDraw.Draw(img).text((off, off), ch, (0, 0, 0), font=f)
    return img

# 목표 행 디스플레이용: ink-bbox 중앙 정렬
def render_display(font_path, ch, size=128):
    px = size * 4
    f = ImageFont.truetype(font_path, int(px * 0.7))
    img = Image.new('L', (px, px), 255)
    d = ImageDraw.Draw(img)
    b = d.textbbox((0, 0), ch, font=f)
    d.text(((px - (b[2] - b[0])) / 2 - b[0], (px - (b[3] - b[1])) / 2 - b[1]), ch, font=f, fill=0)
    return img.resize((size, size), Image.LANCZOS)

render_train('target_font.ttf', REF_CHAR).save('1_train.png')
print('테스트 셋:', GEN_CHARS, '/ 참조 1_train.png 저장')

In [ ]:
# sample.py 시드 고정 패치 — 배치마다 deterministic seed (재현 가능한 측정)
src = open('sample.py').read()
if 'manual_seed' not in src:
    src = src.replace(
        "    while len(all_images) * cfg.batch_size < cfg.num_samples:",
        "    while len(all_images) * cfg.batch_size < cfg.num_samples:\n        th.manual_seed(20260610 + ch_idx)",
    )
    open('sample.py', 'w').write(src)
    print('sample.py 시드 패치 완료')

N = 8                                  # 글자당 샘플 수
gen = ''.join(c * N for c in GEN_CHARS)
open('gen_char_kor.txt', 'w', encoding='utf-8').write(gen)

cfg = f"""model_path: ./trained_models/ema_0.9999_800000.pt
sty_img_path: ./1_train.png
total_txt_file: ./total_kor.txt
gen_txt_file: ./gen_char_kor.txt
stroke_path: ./korean_stroke.txt
img_save_path: ./result_pool
chara_nums: 11172
image_size: 128
num_channels: 128
num_res_blocks: 3
attention_resolutions: "40,20,10"
dropout: 0.1
diffusion_steps: 1000
noise_schedule: linear
use_ddim: true
timestep_respacing: "ddim50"
classifier_free: true
cont_scale: 3.0
sk_scale: 3.0
batch_size: 8
num_samples: {len(gen)}
"""
import os
os.makedirs('cfg', exist_ok=True)
open('cfg/pool.yaml', 'w').write(cfg)
print(f'{len(GEN_CHARS)} 자 x {N} = {len(gen)} 샘플 / batch 8 / ddim50 / cont,sk=3.0 / 학습정합 참조')

In [ ]:
import subprocess, shutil, os, time
if os.path.exists('result_pool'):
    shutil.rmtree('result_pool')
os.makedirs('result_pool')
t0 = time.time()
subprocess.run(['python', 'sample.py', '--cfg_path', 'cfg/pool.yaml'], check=False)
dt = time.time() - t0
pool_pngs = sorted(f for f in os.listdir('result_pool') if f.endswith('.png'))
SEC_PER_SAMPLE = dt / max(len(pool_pngs), 1)
print(f'{len(pool_pngs)} PNG / {dt:.0f}s / {SEC_PER_SAMPLE:.2f}s per 샘플')

## 2. 선택기 — 프로토타입 분류기 (주) + easyocr (보조)

In [ ]:
# 11,172 자 전부를 학습정합 렌더 → 구조 기반 nearest-prototype 분류기
# (생성 글리프가 11,172 자 중 어느 글자에 가장 가까운지 — OCR 무관, 결정론적)
import numpy as np
with open('total_kor.txt', encoding='utf-8') as f:
    ALL_CHARS = f.read().strip()
P = 48
print(f'{len(ALL_CHARS)} 자 프로토타입 렌더 중 (~1분)')
protos = np.zeros((len(ALL_CHARS), P * P), dtype=np.float32)
for i, ch in enumerate(ALL_CHARS):
    im = render_train('target_font.ttf', ch).convert('L').resize((P, P))
    a = 1.0 - np.asarray(im, dtype=np.float32) / 255.0
    n = float(np.linalg.norm(a))
    protos[i] = a.ravel() / n if n > 0 else a.ravel()

def proto_classify(path):
    im = Image.open(path).convert('L').resize((P, P))
    a = 1.0 - np.asarray(im, dtype=np.float32) / 255.0
    n = float(np.linalg.norm(a))
    v = a.ravel() / n if n > 0 else a.ravel()
    return ALL_CHARS[int(np.argmax(protos @ v))]

print('프로토타입 분류기 준비', protos.shape)

In [ ]:
ocr_fn = None
try:
    import easyocr
    _reader = easyocr.Reader(['ko'], gpu=True)
    def _ocr(path, m=48):
        im = Image.open(path).convert('L')
        c = Image.new('L', (im.width + 2 * m, im.height + 2 * m), 255)
        c.paste(im, (m, m))
        return ''.join(_reader.readtext(np.asarray(c), detail=0)).strip()
    ocr_fn = _ocr
    print('easyocr 준비 (보조 선택기)')
except Exception as ex:
    print('easyocr 불가 — 프로토타입 분류기 단독 진행:', ex)

## 3. 판독 → coverage → 비용 외삽 (결정 데이터)

In [ ]:
import math

pool = {}
for r, ch in enumerate(GEN_CHARS):
    es = []
    for s in range(N):
        i = r * N + s
        if i >= len(pool_pngs):
            break
        p = f'result_pool/{pool_pngs[i]}'
        pr = proto_classify(p)
        oc = ocr_fn(p) if ocr_fn else None
        es.append({'path': p, 'proto': pr, 'ocr': oc, 'p_ok': pr == ch, 'o_ok': oc == ch})
    pool[ch] = es
    ph = sum(e['p_ok'] for e in es)
    oh = sum(e['o_ok'] for e in es)
    print(f"{ch}: proto {ph}/{N}  ocr {oh}/{N} | proto 판독: {[e['proto'] for e in es]}")

cov_p = sum(1 for ch in GEN_CHARS if any(e['p_ok'] for e in pool[ch]))
cov_u = sum(1 for ch in GEN_CHARS if any(e['p_ok'] or e['o_ok'] for e in pool[ch]))
print(f'\n=== coverage: proto {cov_p}/{len(GEN_CHARS)} · proto∪ocr {cov_u}/{len(GEN_CHARS)} ===')

# 난이도 클래스별 성공률 → 95% 도달에 필요한 글자당 샘플 수 N95 → 비용 외삽
# 클래스 가중치 (전체 11,172): 받침无 399 / 홑받침 5,586 / 겹받침 5,187
# KS X 1001 2,350 근사 분할: 340 / 1,900 / 110 (흔한 음절은 홑받침 위주)
W_FULL = {'단순(받침无)': 399, '일반(홑받침)': 5586, '복합(겹받침)': 5187}
W_2350 = {'단순(받침无)': 340, '일반(홑받침)': 1900, '복합(겹받침)': 110}

print('\n-- 난이도별 측정 + 외삽 (proto 기준, 24 샘플/클래스 추정치 — 오차 큼) --')
tot_full = tot_2350 = 0.0
feasible = True
for cls, chs in DIFF.items():
    hits = sum(sum(e['p_ok'] for e in pool[ch]) for ch in chs)
    n_obs = len(chs) * N
    p = hits / n_obs
    if p <= 0:
        n95 = None
        feasible = False
        print(f'{cls}: p̂=0.00 ({hits}/{n_obs}) → N=8 로 도달 실패, 외삽 불가')
        continue
    n95 = 1 if p >= 1 else math.ceil(math.log(0.05) / math.log(1 - p))
    tot_full += W_FULL[cls] * n95
    tot_2350 += W_2350[cls] * n95
    print(f'{cls}: p̂={p:.2f} ({hits}/{n_obs}) → 글자당 N95={n95}')

if feasible:
    h_full = tot_full * SEC_PER_SAMPLE / 3600
    h_2350 = tot_2350 * SEC_PER_SAMPLE / 3600
    print(f'\n>>> 전체 11,172 자: ~{tot_full:,.0f} 샘플 ≈ {h_full:.0f} GPU·h (T4)')
    print(f'>>> KS X 1001 2,350 자: ~{tot_2350:,.0f} 샘플 ≈ {h_2350:.1f} GPU·h (T4)')
    print('\n판정 가이드: 2,350 자가 ~25 GPU·h 이하면 프리뷰 폰트 Phase 1 현실권')
else:
    print('\n>>> 일부 클래스 도달 실패 — 해당 난이도는 이 ckpt 로 불가, 부분 문자셋 또는 학습 트랙 검토')

In [ ]:
from PIL import Image, ImageDraw

CELL, PAD, LABELW, TEXTH = 96, 6, 80, 20
W = LABELW + (CELL + PAD) * (N + 1) + PAD
ROWH = CELL + TEXTH + PAD
out = Image.new('RGB', (W, ROWH * len(GEN_CHARS) + PAD), (255, 255, 255))
draw = ImageDraw.Draw(out)
for r, ch in enumerate(GEN_CHARS):
    y = PAD + r * ROWH
    draw.text((4, y + CELL // 2), f'{ch} 목표', fill=(0, 0, 0))
    out.paste(render_display('target_font.ttf', ch, CELL).convert('RGB'), (LABELW, y))
    for s, e in enumerate(pool[ch]):
        x = LABELW + (s + 1) * (CELL + PAD)
        im = Image.open(e['path']).convert('RGB').resize((CELL, CELL))
        out.paste(im, (x, y))
        col = (0, 170, 0) if e['p_ok'] else (220, 60, 60)
        draw.rectangle([x, y, x + CELL, y + CELL], outline=col, width=3)
        draw.text((x + 2, y + CELL + 2), f"P:{e['proto']}", fill=col)
out.save('ocr_select.png')
print('ocr_select.png — 녹색 = 프로토타입 분류가 목표와 일치(선택)')
display(out)

sel = Image.new('RGB', (LABELW + (CELL + PAD) * len(GEN_CHARS) + PAD, CELL + 2 * PAD), (255, 255, 255))
d2 = ImageDraw.Draw(sel)
d2.text((4, CELL // 2), '선택 결과', fill=(0, 0, 0))
for c, ch in enumerate(GEN_CHARS):
    picks = [e['path'] for e in pool[ch] if e['p_ok']]
    x = LABELW + c * (CELL + PAD)
    if picks:
        im = Image.open(picks[0]).convert('RGB').resize((CELL, CELL))
        box = (0, 170, 0)
    else:
        im = Image.new('RGB', (CELL, CELL), (245, 245, 245))
        box = (220, 60, 60)
    sel.paste(im, (x, PAD))
    d2.rectangle([x, PAD, x + CELL, PAD + CELL], outline=box, width=3)
sel.save('selected_font.png')
print('selected_font.png — 글자당 선택 1 샘플 (회색 = 도달 실패)')
display(sel)